# NLP HW2 — самостоятельный инференс на одной T4

Чистый runtime, выберите **T4**, Run all, подтвердите Drive. A100-сессия и её `/content` не нужны. Единственный вход — проверенный inference export на Drive. Код весь в notebook: GitHub/наши скрипты не загружаются. Нет обучения, pretrained weights или исполнения generated-программ.

Зафиксирован `candidate_000082048` из `large_20261023_155405`: локальная numerical validation355/493, не public pass@1. Greedy/max256/batch32 соответствует проверенным validation-настройкам. `EXPORT_PATH=None` при необходимости возвращает автоматический CE-выбор последнего завершённого run. После сохранения на Drive автоматически скачивается `t4_results.zip`, содержащий submission.jsonl и диагностику. Не заявляем180s до реального замера. После завершения вручную отключите T4.

## 1. Настройки и начало отдельного30-минутного бюджета

In [ ]:
import os, sys, time, json, shutil, hashlib, subprocess, zipfile
from pathlib import Path

T4_STARTED_UNIX = time.time()
T4_STARTED_PERF = time.perf_counter()
if "torch" in sys.modules and sys.modules["torch"].cuda.is_initialized():
    raise RuntimeError("Restart session before Run all")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
DRIVE_ROOT = Path("/content/drive/MyDrive/nlp_hw2")
EXPORT_PATH = Path(
    "/content/drive/MyDrive/nlp_hw2/runs/large_20261023_155405/exports/candidate_000082048"
)  # selected by local numerical validation:355/493
INFERENCE_BATCH = 32  # matches fixed512 numerical validation settings
AUTO_DOWNLOAD_RESULTS = True  # download one ZIP containing submission.jsonl and diagnostics
T4_BUDGET_SECONDS = 30 * 60
INFERENCE_DEADLINE_UNIX = T4_STARTED_UNIX + T4_BUDGET_SECONDS
LOCAL_EXPORT = Path("/content/hw2_T4_export")
OUTPUT = Path("/content/hw2_T4_outputs")
print("T4 clock starts now; GPU time before Run all is not measured.")


## 2. Drive, выбор экспорта, локальная копия

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
if EXPORT_PATH is None:
    completed = []
    for path in (DRIVE_ROOT / "runs").glob("*/completion.json"):
        info = json.loads(path.read_text())
        completed.append((info["finished_utc"], info, path))
    if not completed:
        raise FileNotFoundError(
            "No verified completed A100 run; set explicit EXPORT_PATH only for a verified portable export"
        )
    _, completion, _ = max(completed, key=lambda x: x[0])
    EXPORT_PATH = Path(completion["selected_export"]["export_path"])
EXPORT_PATH = Path(EXPORT_PATH)
if not (EXPORT_PATH / "bundle_manifest.json").exists():
    raise FileNotFoundError("Export is not complete")
manifest = json.loads((EXPORT_PATH / "bundle_manifest.json").read_text())
print(
    "Selected:",
    EXPORT_PATH,
    "provisional CE:",
    manifest["candidate"]["ce"],
    "identity:",
    manifest["identity"],
)
if time.time() > INFERENCE_DEADLINE_UNIX - 180:
    raise TimeoutError("Insufficient T4 budget")
t = time.perf_counter()
if LOCAL_EXPORT.exists():
    raise FileExistsError("Restart clean runtime; do not reuse stale local weights")
shutil.copytree(EXPORT_PATH, LOCAL_EXPORT)
COPY_EXPORT_SECONDS = time.perf_counter() - t
print("Local copy seconds:", COPY_EXPORT_SECONDS)


## 3. Зависимости
CUDA-enabled torch не переустанавливается. Время установки — отдельно от inference pipeline.

In [ ]:
t = time.perf_counter()
remaining = INFERENCE_DEADLINE_UNIX - time.time()
if remaining < 180:
    raise TimeoutError("Insufficient T4 budget for installation and inference")
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--disable-pip-version-check",
        "-q",
        "-r",
        str(LOCAL_EXPORT / "requirements-inference.txt"),
    ],
    check=True,
    timeout=max(1, min(600, remaining - 120)),
)
DEPENDENCY_SECONDS = time.perf_counter() - t
print("Dependency seconds:", DEPENDENCY_SECONDS)


## 4. Полный standalone код инференса
Таймер pipeline начинается перед импортами torch/Transformers в этой ячейке, включает проверку файлов, CUDA, загрузку, warmup, всю генерацию и запись outputs. Drive copy и установка показаны отдельно. Generation измеряется с CUDA synchronization; warmup не скрывается. Нужна ровно одна видимая T4. В Colab generated-код не исполняется.

In [ ]:
IN_NOTEBOOK = True
import os, sys, time, json, hashlib, math, platform, importlib.metadata
from pathlib import Path

PIPELINE_STARTED = time.perf_counter()
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
import torch
from transformers import BartForConditionalGeneration
from tokenizers import Tokenizer
from contextlib import nullcontext


def file_sha(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        while chunk := f.read(8 * 1024 * 1024):
            h.update(chunk)
    return h.hexdigest()


def infer_generate(model, tokenizer, texts, batch_size=64, max_new_tokens=256):
    records = []
    model.eval()
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()
    started = time.perf_counter()
    for offset in range(0, len(texts), batch_size):
        if time.time() + 30 >= globals().get("INFERENCE_DEADLINE_UNIX", float("inf")):
            raise TimeoutError("T4 wall-clock budget reached")
        encoded = [
            [1] + x.ids + [2] for x in tokenizer.encode_batch(texts[offset : offset + batch_size])
        ]
        if max(map(len, encoded)) > 192:
            raise ValueError("Source exceeds192; no silent truncation")
        width = math.ceil(max(map(len, encoded)) / 8) * 8
        ids = torch.zeros((len(encoded), width), dtype=torch.long, device="cuda")
        for i, row in enumerate(encoded):
            ids[i, : len(row)] = torch.tensor(row, device="cuda")
        with torch.inference_mode(), torch.autocast("cuda", dtype=torch.float16):
            outputs = model.generate(
                input_ids=ids,
                attention_mask=ids.ne(0),
                do_sample=False,
                num_beams=1,
                use_cache=True,
                max_new_tokens=max_new_tokens,
                forced_eos_token_id=None,
                pad_token_id=0,
                eos_token_id=2,
                decoder_start_token_id=1,
            )
        for i, row in enumerate(outputs.cpu().tolist()):
            row = row[1:]
            eos = 2 in row
            if eos:
                row = row[: row.index(2) + 1]
            records.append(
                {
                    "index": offset + i,
                    "question_sha256": hashlib.sha256(texts[offset + i].encode()).hexdigest(),
                    "code": tokenizer.decode(row, skip_special_tokens=True),
                    "tokens": len(row),
                    "eos": eos,
                    "hit_limit": not eos and len(row) >= max_new_tokens,
                }
            )
    torch.cuda.synchronize()
    return records, {
        "generation_seconds": time.perf_counter() - started,
        "examples": len(records),
        "peak_allocated_gib": torch.cuda.max_memory_allocated() / 2**30,
        "peak_reserved_gib": torch.cuda.max_memory_reserved() / 2**30,
        "batch_size": batch_size,
    }


def inference_run(bundle, output, batch_size=64, require_t4=True):
    bundle, output = Path(bundle), Path(output)
    output.mkdir(parents=True, exist_ok=True)
    if not torch.cuda.is_available() or torch.cuda.device_count() != 1:
        raise RuntimeError("Exactly one visible CUDA GPU required")
    gpu = torch.cuda.get_device_name(0)
    if require_t4 and "T4" not in gpu:
        raise RuntimeError("Select T4; this notebook is the T4 measurement")
    torch.set_num_threads(4)
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    versions = {
        n: importlib.metadata.version(n)
        for n in ["torch", "transformers", "tokenizers", "numpy", "safetensors"]
    }
    print(
        "GPU:",
        gpu,
        "CUDA:",
        torch.version.cuda,
        "Python:",
        sys.version,
        "versions:",
        versions,
        flush=True,
    )
    manifest = json.loads((bundle / "bundle_manifest.json").read_text())
    for name, digest in manifest["asset_sha256"].items():
        path = (bundle / name).resolve()
        if bundle.resolve() not in path.parents or file_sha(path) != digest:
            raise IOError("Export checksum mismatch: " + name)
    inputs = [json.loads(x) for x in (bundle / "test_questions.jsonl").read_text().splitlines()]
    if len(inputs) != 1919 or any(
        set(r) != {"index", "question"} or r["index"] != i or not isinstance(r["question"], str)
        for i, r in enumerate(inputs)
    ):
        raise ValueError("Expected1919 original indexed question-only rows")
    texts = [r["question"] for r in inputs]
    torch.cuda.synchronize()
    t = time.perf_counter()
    model = (
        BartForConditionalGeneration.from_pretrained(
            bundle / "model", local_files_only=True, attn_implementation="sdpa"
        )
        .cuda()
        .eval()
    )
    tokenizer = Tokenizer.from_file(str(bundle / "tokenizer.json"))
    if [tokenizer.token_to_id(s) for s in ["<pad>", "<bos>", "<eos>", "<unk>"]] != [0, 1, 2, 3]:
        raise ValueError("Special tokens differ")
    torch.cuda.synchronize()
    loading = time.perf_counter() - t
    _, warm = infer_generate(model, tokenizer, texts[:8], 8)
    torch.cuda.empty_cache()
    records, timing = infer_generate(model, tokenizer, texts, batch_size)
    if len(records) != 1919:
        raise ValueError("Output count differs")
    for i, (record, text) in enumerate(zip(records, texts)):
        if (
            record["index"] != i
            or record["question_sha256"] != hashlib.sha256(text.encode()).hexdigest()
        ):
            raise ValueError("Input/output alignment differs")
    (output / "predictions.json").write_text(json.dumps(records, ensure_ascii=False, indent=2))
    submission = output / "submission.jsonl"
    with submission.open("w", encoding="utf-8") as f:
        for record in records:
            f.write(json.dumps({"code": record["code"]}, ensure_ascii=False) + "\n")
    parsed = [json.loads(line) for line in submission.read_text().splitlines()]
    assert len(parsed) == 1919 and all(
        set(r) == {"code"} and r["code"] == x["code"] for r, x in zip(parsed, records)
    )
    settings = {
        "do_sample": False,
        "num_beams": 1,
        "max_new_tokens": 256,
        "batch_size": batch_size,
        "precision": "float16",
        "sdpa": True,
        "kv_cache": True,
        "forced_eos_token_id": None,
    }
    (output / "generation_parameters.json").write_text(json.dumps(settings, indent=2))
    (output / "submission_validation.json").write_text(
        json.dumps(
            {
                "count": 1919,
                "only_code_key": True,
                "original_bundled_order_verified": True,
                "official_pass1_file_separately_compared": False,
                "input_sha256": file_sha(bundle / "test_questions.jsonl"),
                "submission_sha256": file_sha(submission),
                "programs_edited_or_executed": False,
                "bot_submission_sent": False,
            },
            indent=2,
        )
    )
    timing.update(
        {
            "gpu": gpu,
            "versions": versions,
            "load_model_tokenizer_seconds": loading,
            "warmup_seconds": warm["generation_seconds"],
            "warmup_examples": 8,
            "full_pipeline_seconds": time.perf_counter() - PIPELINE_STARTED,
            "pipeline_includes": "imports in this cell/process, checksums, CUDA initialization, model/tokenizer load, warmup, full generation, output writes up to timing.json",
            "pipeline_excludes": "Drive mount/copy and dependency installation; reported separately by notebook",
            "generation_under_180_seconds": timing["generation_seconds"] <= 180,
            "full_pipeline_under_180_seconds": time.perf_counter() - PIPELINE_STARTED <= 180,
            "T4_measured": "T4" in gpu,
            "public_pass_at_1": None,
            "identity": manifest["identity"],
        }
    )
    (output / "timing.json").write_text(json.dumps(timing, indent=2))
    print(json.dumps(timing, indent=2), flush=True)
    return timing


if __name__ == "__main__" and "IN_NOTEBOOK" not in globals():
    import argparse

    p = argparse.ArgumentParser()
    p.add_argument("--bundle", type=Path, required=True)
    p.add_argument("--output", type=Path, required=True)
    p.add_argument("--batch", type=int, default=64)
    a = p.parse_args()
    inference_run(a.bundle, a.output, a.batch)

import contextlib


class Tee:
    def __init__(self, *streams):
        self.streams = streams

    def write(self, text):
        for stream in self.streams:
            stream.write(text)
        return len(text)

    def flush(self):
        for stream in self.streams:
            stream.flush()


OUTPUT.mkdir(parents=True, exist_ok=True)
with (OUTPUT / "inference.log").open("w", buffering=1) as log:
    with contextlib.redirect_stdout(Tee(sys.stdout, log)), contextlib.redirect_stderr(
        Tee(sys.stderr, log)
    ):
        TIMING = inference_run(LOCAL_EXPORT, OUTPUT, INFERENCE_BATCH, require_t4=True)
TIMING.update(
    {
        "dependency_install_seconds": DEPENDENCY_SECONDS,
        "copy_export_seconds": COPY_EXPORT_SECONDS,
        "notebook_total_before_Drive_save_seconds": time.perf_counter() - T4_STARTED_PERF,
    }
)
(OUTPUT / "timing.json").write_text(json.dumps(TIMING, indent=2))


## 5. Проверенная запись на Drive и что скачать
Боту только `submission.jsonl`:1919 строк, один ключ code. Для локальной проверки скачайте `t4_results.zip` и выполненный notebook. Оплата runtime не останавливается завершением Python: отключите T4 вручную.

In [ ]:
def verified_copy(src, dst):
    dst.parent.mkdir(parents=True, exist_ok=True)
    tmp = dst.with_name(dst.name + ".upload")
    shutil.copyfile(src, tmp)
    if file_sha(src) != file_sha(tmp):
        raise IOError("Drive output checksum mismatch")
    tmp.replace(dst)


result_name = "T4_" + time.strftime("%Y%m%d_%H%M%S")
DRIVE_OUTPUT = DRIVE_ROOT / "inference" / result_name
started = time.perf_counter()
for p in OUTPUT.glob("*"):
    if p.is_file():
        verified_copy(p, DRIVE_OUTPUT / p.name)
TIMING["Drive_output_copy_seconds"] = time.perf_counter() - started
TIMING["notebook_total_seconds"] = time.perf_counter() - T4_STARTED_PERF
(OUTPUT / "timing.json").write_text(json.dumps(TIMING, indent=2))
verified_copy(OUTPUT / "timing.json", DRIVE_OUTPUT / "timing.json")
with zipfile.ZipFile(OUTPUT / "t4_results.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for p in OUTPUT.glob("*"):
        if p.is_file() and p.name != "t4_results.zip":
            z.write(p, p.name)
verified_copy(OUTPUT / "t4_results.zip", DRIVE_OUTPUT / "t4_results.zip")
print("Saved and checksummed:", DRIVE_OUTPUT)
print(
    "generation seconds:",
    TIMING["generation_seconds"],
    "full pipeline seconds:",
    TIMING["full_pipeline_seconds"],
)
from IPython.display import display, FileLink

os.chdir(OUTPUT)
display(FileLink("submission.jsonl"))
display(FileLink("t4_results.zip"))

# Browser download starts only after the complete ZIP was verified on Drive.
if AUTO_DOWNLOAD_RESULTS:
    from google.colab import files

    try:
        files.download(str(OUTPUT / "t4_results.zip"))
    except Exception as exc:
        print("Browser download did not complete:", repr(exc))
        print("Download the verified copy from Drive:", DRIVE_OUTPUT / "t4_results.zip")
print("T4 МОЖНО ОТКЛЮЧАТЬ ВРУЧНУЮ. Боту нужен только submission.jsonl из ZIP.")
